# 02 - Hybrid Resampling & CTGAN (Module-First)

Notebook nay chi dieu phoi pipeline can bang class thong qua module da co san.

Luong chay:
1. Nap `01_processed_features.parquet`
2. Tao `ClassBalanceConfig`
3. Goi `build_balanced_dataset_from_path(...)`
4. Kiem tra `02_balanced_training_data.parquet` + class distribution truoc/sau

Business logic can bang du lieu nam trong module:
- `src.rl.data_balance.pipeline`

In [ ]:
from pathlib import Path
import json
import pandas as pd

from src.rl.data_balance.pipeline import ClassBalanceConfig, build_balanced_dataset_from_path

In [ ]:
# ==== Config ====
INPUT_PATH = Path('/workspace/ai-core/notebooks/01_processed_features.parquet')
OUTPUT_PATH = Path('/workspace/ai-core/notebooks/02_balanced_training_data.parquet')
REPORT_PATH = Path('/workspace/ai-core/notebooks/02_balanced_training_report.json')

config = ClassBalanceConfig(
    random_seed=42,
    window_size=12,
    majority_cap_multiplier=2.5,
    transition_multiplier=1.30,
    duplicate_multiplier=0.20,
    duplicate_mae_threshold=1e-3,
    synthetic_rows_class4=50_000,
    synthetic_rows_class5=20_000,
    synthetic_noise_pct=0.02,
    use_ctgan=True,
    output_path=str(OUTPUT_PATH),
    report_path=str(REPORT_PATH),
)

print('Input path :', INPUT_PATH)
print('Output path:', OUTPUT_PATH)
print('Report path:', REPORT_PATH)

In [ ]:
assert INPUT_PATH.exists(), f'Missing input parquet: {INPUT_PATH}'

balanced_df, report = build_balanced_dataset_from_path(
    input_path=INPUT_PATH,
    config=config,
    output_path=OUTPUT_PATH,
    report_path=REPORT_PATH,
)

print('Balanced shape:', balanced_df.shape)
print('Saved parquet :', OUTPUT_PATH)
print('Saved report  :', REPORT_PATH)

report_dict = report.to_dict()
print('Before counts:', report_dict.get('before_counts'))
print('After counts :', report_dict.get('after_counts'))

In [ ]:
# Quick validation
saved_df = pd.read_parquet(OUTPUT_PATH)
print('Reloaded shape:', saved_df.shape)
print('Class counts:')
print(saved_df['congestion_level'].value_counts().sort_index())

if REPORT_PATH.exists():
    payload = json.loads(REPORT_PATH.read_text(encoding='utf-8'))
    print('Report keys:', sorted(payload.keys()))
    print('Stage counts:', payload.get('stage_counts', {}))

saved_df.head(5)

In [ ]:
# Legacy manual balancing cell replaced by module pipeline.
# Keep this cell as a placeholder for optional custom experiments.

In [ ]:
# Placeholder: add optional diagnostics/plots here if needed.

In [ ]:
# Placeholder: keep notebook-thin. Core balancing logic stays in src.rl.data_balance.pipeline.

In [ ]:
# Optional: visualize class distributions using report_dict/saved_df if needed.